# 3 · Results: posteriors, calibration, type confusion, a real event

Uses the notebook-2 model by default; if a production checkpoint
(`model/lean_npe_v3/best_model.pth`) is present it is used instead, so the same
cells show real results.

1. posteriors for simulated events vs. the truth;
2. calibration: coverage and SBC (are the 90% intervals right 90% of the time?);
3. the **type-confusion probe** that motivated per-class models;
4. a real GWTC event through `infer()`: detector-frame masses, the d² prior, model selection;
5. the production certification report.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# run from the repo root so relative paths (configs/, data/, model/) resolve
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
for p in (ROOT, ROOT / "src", ROOT / "experiments"):  # experiments/: remix_data imports delay_aug
    sys.path.insert(0, str(p))
import logging; logging.getLogger("bilby").setLevel(logging.WARNING)

DATA = ROOT / "data" / "notebook_demo"      # small demo dataset (git-ignored)
MODEL_DIR = ROOT / "model" / "notebook_demo" # small demo model (git-ignored)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("repo:", ROOT, "| device:", DEVICE)

from ahsd.inference.pipeline import load_model
from ahsd.models.lean_npe import PARAM_NAMES
from experiments.remix_data import RemixDataset
J = {n: i for i, n in enumerate(PARAM_NAMES)}
CKPT = ROOT / "model" / "lean_npe_v3" / "best_model.pth"
CKPT = CKPT if CKPT.exists() else MODEL_DIR / "best_model.pth"
model, meta = load_model(str(CKPT), DEVICE)
print("model:", CKPT.relative_to(ROOT), "| epoch", meta["model_epoch"], "| trained on", meta["train_event_types"])

## 1. Posteriors on simulated validation events

In [ ]:
val_ds = RemixDataset(str(DATA / "memmap" / "validation"), remix=False, seed=1234, return_asd_bands=True)
items = [val_ds[i] for i in range(len(val_ds))]
strain = torch.stack([it[0] for it in items]).float().to(DEVICE)
truth = torch.stack([it[1][0] for it in items]).numpy()            # rank-0 (loudest) signal
asd = torch.stack([it[4] for it in items]).float().to(DEVICE)
with torch.no_grad():
    S = torch.cat([model.sample_posterior(strain[i:i + 32], n_samples=400, asd_bands=asd[i:i + 32])
                   for i in range(0, len(strain), 32)]).cpu().numpy()     # [events, samples, 11]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
for a, name in zip(ax, ("mass_1", "mass_2", "luminosity_distance", "geocent_time")):
    j = J[name]; med = np.median(S[:, :, j], 1); lo, hi = np.quantile(S[:, :, j], [0.05, 0.95], axis=1)
    a.errorbar(truth[:, j], med, yerr=[med - lo, hi - med], fmt=".", alpha=0.4, ms=3)
    lim = [truth[:, j].min(), truth[:, j].max()]; a.plot(lim, lim, "k--", lw=1); a.set(title=name, xlabel="truth", ylabel="median ± 90%")
plt.tight_layout(); plt.show()

## 2. Calibration: coverage and SBC

If the posteriors are calibrated, the truth falls inside the 90% interval 90% of the time,
and its rank among the samples is uniform (simulation-based calibration). NLL alone does
not show this — a model can lower NLL by becoming overconfident.

In [ ]:
ranks = (S < truth[:, None, :]).mean(1)
cov90 = ((ranks > 0.05) & (ranks < 0.95)).mean(0)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].bar(PARAM_NAMES, cov90); ax[0].axhline(0.9, c="k", ls="--"); ax[0].set(title="90% coverage", ylim=(0, 1)); ax[0].tick_params(axis="x", rotation=60)
for j, name in enumerate(PARAM_NAMES):
    ax[1].plot(np.sort(ranks[:, j]), np.linspace(0, 1, len(ranks)), lw=1, label=name)
ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set(title="SBC / P-P (diagonal = calibrated)", xlabel="rank of truth", ylabel="CDF")
ax[1].legend(fontsize=6, ncol=2); plt.tight_layout(); plt.show()

## 3. Type confusion

A single model for BBH + NSBH + BNS can put posterior mass on the **wrong source class**
(they share chirp masses). Distance ranges differ by class, so this biases distance:
for the production mixed model, BNS posteriors put 38% of their samples on other classes
and BNS distances came out 2× too far — restricting to the right class removed most of the
bias. That is why there are per-class models (`infer.py --event-type`).

In [ ]:
def cls(m1, m2):
    return np.where(m1 < 3, 0, np.where(m2 < 3, 1, 2))     # 0 BNS, 1 NSBH, 2 BBH
tt, st = cls(truth[:, 0], truth[:, 1]), cls(S[:, :, 0], S[:, :, 1])
names = ["BNS", "NSBH", "BBH"]
print("true class | posterior mass on  BNS  NSBH   BBH | distance true/pred (all samples -> right class only)")
for c_ in (2, 1, 0):
    k = tt == c_
    if k.sum() < 3:
        continue
    frac = [(st[k] == u).mean() for u in (0, 1, 2)]
    d, dt = S[k][:, :, J["luminosity_distance"]], truth[k, J["luminosity_distance"]]
    all_b = np.exp(np.mean(np.log(dt) - np.log(np.median(d, 1))))
    right = [np.log(dt[e]) - np.log(np.median(d[e][st[k][e] == c_])) for e in range(k.sum()) if (st[k][e] == c_).sum() >= 20]
    print(f"{names[c_]:10s} |                 {frac[0]:.2f}  {frac[1]:.2f}  {frac[2]:.2f} | x{all_b:.2f} -> x{np.exp(np.mean(right)) if right else float('nan'):.2f}")

## 4. A real event: GW150914

`infer()` fetches the strain from GWOSC, whitens it exactly as in training, and returns a
`PosteriorResult`. Two conventions matter when comparing with published values:
- published masses are **source frame**; the model measures **detector frame**, m_det = m_src (1+z);
- published distances use a **d² prior**; the model was trained uniform in d, so reweight.

`event_type` selects the per-class model when one exists (`diagnostics["model_selection"]`).

In [ ]:
from ahsd.inference import infer
from astropy.cosmology import Planck15, z_at_value
import astropy.units as u
try:
    res = infer(event="GW150914", event_type="BBH", model=str(CKPT), num_samples=4000, device="cpu")
    pub = {"m1": 38.8, "m2": 33.4, "dL": 440}
    z = float(z_at_value(Planck15.luminosity_distance, pub["dL"] * u.Mpc))
    med = res.median; rw = res.reweight_distance_prior(target="comoving_d2")
    q = res.samples[:, 1] / res.samples[:, 0]
    print(f"m1 {med['mass_1']:.1f} (published {pub['m1'] * (1 + z):.1f}, detector frame)")
    print(f"m2 {med['mass_2']:.1f} (published {pub['m2'] * (1 + z):.1f}, detector frame)")
    print(f"q  {np.median(q):.2f}  90% [{np.quantile(q, .05):.2f}, {np.quantile(q, .95):.2f}]  (published {pub['m2'] / pub['m1']:.2f})")
    print(f"dL {med['luminosity_distance']:.0f} Mpc (training prior) -> {rw.median['luminosity_distance']:.0f} (d^2 prior), published {pub['dL']}")
    print("railing", f"{res.diagnostics['boundary_railing_frac']:.1%}", "| confidence", res.diagnostics.get("verdict", {}).get("confidence"),
          "| model selection", res.diagnostics["model_selection"]["chosen"])
    fig, ax = plt.subplots(1, 3, figsize=(13, 3))
    for a, j, ref in zip(ax, (J["mass_1"], J["mass_2"], J["luminosity_distance"]), (pub["m1"] * (1 + z), pub["m2"] * (1 + z), pub["dL"])):
        a.hist(res.samples[:, j], bins=60, histtype="step"); a.axvline(ref, c="k", ls="--"); a.set(title=PARAM_NAMES[j])
    plt.tight_layout(); plt.show()
except Exception as e:
    print("real-event inference needs network access to GWOSC:", type(e).__name__, e)

## 5. The production certification report

`scripts/validate_checkpoint.py` writes `analysis/ci/<model>/report.json` (+ a self-contained
HTML report): population calibration gates, sharpness vs dynesty, per-class distance, a
synthetic basin probe, and the real GWTC events of the model's training classes.

In [ ]:
rep = ROOT / "analysis" / "ci" / "lean_npe_v3" / "report.json"
if rep.exists():
    r = json.load(open(rep))
    print(f"{r['meta']['model']} | epoch {r['meta']['epoch']} | val NLL {r['meta']['val_nll']:.3f}\n")
    for g in r["gates"]:
        print(("PASS " if g["pass"] else "FAIL ") + g["gate"][:70].ljust(70), "|", str(g["detail"])[:60])
    print("\nreal events:")
    for k, v in (r.get("gw150914") or {}).items():
        print(f"  {k}: {v[:150]}")
else:
    print("no production report found at", rep)